<div style="background-color:#EAEAEA; padding:20px; border-left:5px solid #6C757D; border-radius:6px;">
<table style="width:100%; border:none;"><tr style="border:none;"><td style="border:none; vertical-align:top;">
<h1 style="font-size:32px; margin-top:0;">Master's Thesis</h1><hr style="margin:16px 0 22px 0;">
<p style="font-size:22px; line-height:1.5; margin:0;"><strong>Master's Degree in Advanced Physics</strong> - <strong>Universitat de València</strong></p>
<p style="font-size:17px; margin-top:28px; margin-bottom:6px;">This notebook is part of the <strong>Master's Thesis (MSc Dissertation)</strong>:</p>
<div style="font-size:25px; font-weight:700; line-height:1.3; margin-top:14px; margin-bottom:26px;">Fast Simulation of Neutrino Oscillations in Matter</div>
<p style="font-size:14px; line-height:1.55;"><strong>Author</strong><br>Juan Ramon Diaz Santos - <a href="mailto:diazjuan@alumni.uv.es">diazjuan@alumni.uv.es</a></p>
<p style="font-size:14px; line-height:1.55;"><strong>Supervisors</strong><br>Roberto Ruiz de Austri Bazan — <a href="mailto:rruiz@ific.uv.es">rruiz@ific.uv.es</a><br>Michele Lucente — <a href="mailto:michele.lucente@unibo.it">michele.lucente@unibo.it</a></p>
<p style="font-size:14px; line-height:1.55; margin-bottom:0;"><strong>Date</strong><br>September 2026</p></td>
<td style="border:none; width:230px; padding-left:25px; text-align:right; vertical-align:top;"><img src="../../../../logo_uv.png" alt="Universitat de València" style="width:200px; margin-top:5px;"></td>
</tr></table></div>

# Bahcall Canonical Data Generator

Downloads official Bahcall tables and generates the canonical density, radial-production, total-flux, energy-spectrum and benchmark-probability products used by TPeanuts.

## 1. Imports and paths

In [1]:
from pathlib import Path
from io import BytesIO, StringIO
import hashlib
import json
import re
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the TPeanuts project root")


def fetch(url: str, *, timeout: int = 60) -> bytes:
    request = urllib.request.Request(url, headers={"User-Agent": "TPeanuts/solar-data"})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return response.read()


def download(url: str, path: Path, *, overwrite: bool = False) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    if overwrite or not path.exists():
        path.write_bytes(fetch(url))
    return path


PROJECT_ROOT = find_project_root(Path.cwd())
SOLAR_DATA = PROJECT_ROOT / "data" / "solar"

## 2. Download official raw data

In [2]:
PROVIDER_DIR = SOLAR_DATA / "bahcall"
RAW_DIR = PROVIDER_DIR / "raw"
for category in ("density", "production", "flux", "spectrum", "probability", "metadata"):
    (PROVIDER_DIR / category).mkdir(parents=True, exist_ok=True)

URLS = {
    "bp2000_neordered.output": "https://www.sns.ias.edu/~jnb/SNdata/Export/BP2000/neordered.output",
    "bp2000_nsterile.output": "https://www.sns.ias.edu/~jnb/SNdata/Export/BP2000/nsterile.output",
    "bp2004flux.dat": "https://www.sns.ias.edu/~jnb/SNdata/Export/BP2004/bp2004flux.dat",
    "ppenergytab": "https://www.sns.ias.edu/~jnb/SNdata/Export/PPenergyspectrum/ppenergytab",
    "hepspectrum.dat": "https://www.sns.ias.edu/~jnb/SNdata/Export/Hepspectrum/hepspectrum.dat",
    "b8spectrum.txt": "https://www.sns.ias.edu/~jnb/SNdata/Export/B8spectrum/b8spectrum.txt",
    "n13.dat": "https://www.sns.ias.edu/~jnb/SNdata/Export/CNOspectra/n13.dat",
    "o15.dat": "https://www.sns.ias.edu/~jnb/SNdata/Export/CNOspectra/o15.dat",
    "f17.dat": "https://www.sns.ias.edu/~jnb/SNdata/Export/CNOspectra/f17.dat",
    "PNU98LMA.dat": "https://www.sns.ias.edu/~jnb/SNdata/Export/MSWsurvival/PNU98LMA.dat",
    "PNU98SMA.dat": "https://www.sns.ias.edu/~jnb/SNdata/Export/MSWsurvival/PNU98SMA.dat",
    "PNU98LOW.dat": "https://www.sns.ias.edu/~jnb/SNdata/Export/MSWsurvival/PNU98LOW.dat",
}
raw_paths = {name: download(url, RAW_DIR / name) for name, url in URLS.items()}
display(pd.DataFrame([{"file": p.name, "bytes": p.stat().st_size} for p in raw_paths.values()]))

,file,bytes
0,bp2000_neordered.output,60942
1,bp2000_nsterile.output,87748
2,bp2004flux.dat,78214
3,ppenergytab,1476
4,hepspectrum.dat,26000
5,b8spectrum.txt,55467
6,n13.dat,5200
7,o15.dat,13000
8,f17.dat,13000
9,PNU98LMA.dat,13901


## 3. Convert and validate

In [3]:
def numeric_rows(path: Path, ncols: int) -> np.ndarray:
    rows = []
    for line in path.read_text(encoding="utf-8", errors="replace").splitlines():
        tokens = line.split()
        if len(tokens) != ncols:
            continue
        try:
            rows.append([float(token) for token in tokens])
        except ValueError:
            pass
    return np.asarray(rows, dtype=float)


ne = numeric_rows(raw_paths["bp2000_neordered.output"], 2)
msw = numeric_rows(raw_paths["bp2000_nsterile.output"], 2)
radius = np.union1d(ne[:, 0], msw[:, 0])
ne_mol = 10.0 ** np.interp(radius, ne[:, 0], ne[:, 1])
msw_mol = 10.0 ** np.interp(radius, msw[:, 0], msw[:, 1])
nn_mol = np.maximum(2.0 * (ne_mol - msw_mol), 0.0)
density = pd.DataFrame({"radius": radius, "electron_density_mol_cm3": ne_mol, "neutron_density_mol_cm3": nn_mol})
density = density.loc[density["radius"] <= 1.0].copy()  # canonical r/R_sun domain
density.to_csv(PROVIDER_DIR / "density" / "bp2000_density_electron_neutron_monotonic.csv", index=False)

bp04 = numeric_rows(raw_paths["bp2004flux.dat"], 13)
columns = ["radius", "temperature_MK", "density_log_10", "shell_mass_solar", "be7_mass_fraction", "pp fraction", "8B fraction", "13N fraction", "15O fraction", "17F fraction", "7Be fraction", "pep fraction", "hep fraction"]
production = pd.DataFrame(bp04, columns=columns)
production.to_csv(PROVIDER_DIR / "production" / "bp2004_production.csv", index=False)

flux_values = [5.938, 1.401e-2, 7.875e-7, 4.857e-1, 5.822e-4, 5.712e-2, 5.031e-2, 5.912e-4]
flux = pd.DataFrame({"fraction": ["pp", "pep", "hep", "7Be", "8B", "13N", "15O", "17F"], "flux": np.asarray(flux_values) * 1.0e10})
flux.to_csv(PROVIDER_DIR / "flux" / "fluxes_bahcall_bp2004.csv", index=False)

spectra = {"8B": numeric_rows(raw_paths["b8spectrum.txt"], 4)[:, :2]}
pp = numeric_rows(raw_paths["ppenergytab"], 8).reshape(-1, 2)
spectra["pp"] = pp
for source, filename in {"hep": "hepspectrum.dat", "13N": "n13.dat", "15O": "o15.dat", "17F": "f17.dat"}.items():
    spectra[source] = numeric_rows(raw_paths[filename], 2)
for source, values in spectra.items():
    values = values[np.argsort(values[:, 0])]
    norm = np.trapz(values[:, 1], x=values[:, 0])
    pd.DataFrame({"Energy": values[:, 0], "Spectrum": values[:, 1] / norm}).to_csv(PROVIDER_DIR / "spectrum" / f"spectrum_{source}.csv", index=False)

for solution in ("LMA", "SMA", "LOW"):
    values = numeric_rows(raw_paths[f"PNU98{solution}.dat"], 4)
    pd.DataFrame(values, columns=["energy_MeV", "day", "day_night_average", "night"]).to_csv(PROVIDER_DIR / "probability" / f"msw_1998_{solution.lower()}.csv", index=False)

metadata = {"provider": "Bahcall Solar Neutrino Software and Data", "url": "https://www.sns.ias.edu/~jnb/SNdata/sndata.html", "default": False, "note": "TPeanuts supports this BP2000/BP2004 combination; the configured default is Zenodo SF-III AGSS09."}
(PROVIDER_DIR / "metadata" / "source.json").write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
display(density.head(), production.head(), flux)

,radius,electron_density_mol_cm3,neutron_density_mol_cm3
0,0.001000,101.981172,50.136257
1,0.008190,101.981172,50.168131
2,0.008317,101.969432,50.155274
3,0.008446,101.957693,50.142418
4,0.008577,101.945955,50.129565


,radius,temperature_MK,density_log_10,shell_mass_solar,be7_mass_fraction,pp fraction,8B fraction,13N fraction,15O fraction,17F fraction,7Be fraction,pep fraction,hep fraction
0,0.00649,15.702,2.011,0.000030,1.737000e-11,0.000189,0.002375,0.001914,0.002178,0.002241,0.000951,0.000298,0.000077
1,0.00659,15.701,2.011,0.000001,1.737000e-11,0.000009,0.000111,0.000090,0.000102,0.000105,0.000044,0.000014,0.000004
2,0.00670,15.701,2.011,0.000001,1.736000e-11,0.000009,0.000116,0.000094,0.000107,0.000110,0.000047,0.000015,0.000004
3,0.00680,15.700,2.011,0.000002,1.736000e-11,0.000010,0.000121,0.000098,0.000111,0.000115,0.000049,0.000015,0.000004
4,0.00690,15.700,2.011,0.000002,1.735000e-11,0.000010,0.000127,0.000102,0.000116,0.000120,0.000051,0.000016,0.000004


,fraction,flux
0,pp,5.938000e+10
1,pep,1.401000e+08
2,hep,7.875000e+03
3,7Be,4.857000e+09
4,8B,5.822000e+06
5,13N,5.712000e+08
6,15O,5.031000e+08
7,17F,5.912000e+06


## 4. Validation

In [4]:
assert (density[["electron_density_mol_cm3", "neutron_density_mol_cm3"]] >= 0).all().all()
assert set(flux["fraction"]) == {"pp", "pep", "hep", "7Be", "8B", "13N", "15O", "17F"}
assert all(abs(np.trapz(v[:, 1] / np.trapz(v[:, 1], x=v[:, 0]), x=v[:, 0]) - 1) < 1e-10 for v in spectra.values())
print("Bahcall canonical products validated.")

Bahcall canonical products validated.
